# Wafer Open-Set Defect Classification

**Objective:** Train a 3-class ResNet18 on known wafer defects, then detect unknown defects (DIE_CRACK, DIE_INK) via post-hoc OOD rejection — not via a 4th classifier output.

**Final method:** Diagonal-covariance Mahalanobis + z-scored Energy–Mahalanobis **soft fusion** (best balanced open-set F1 on validation).

## 2. Imports and configuration


In [1]:
# --- 1. Imports & configuration ---
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

try:
    import wandb
    WANDB_AVAILABLE = True
except Exception:
    WANDB_AVAILABLE = False

# Paths
DATA_ROOT = Path('./Data')
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'
TEST_DIR = DATA_ROOT / 'test'
OUT_DIR = Path('./outputs')
OUT_DIR.mkdir(exist_ok=True)
MODEL_PATH = Path('./model.pth')
BEST_CKPT_PATH = OUT_DIR / 'best_model.pth'

# Classes — unknown is label 3 at evaluation only (never trained)
KNOWN_CLASSES = ['DIE_BROKEN', 'NORMAL', 'NO_DIE']
UNKNOWN_CLASSES = ['DIE_CRACK', 'DIE_INK']
CLASS_TO_IDX = {'DIE_BROKEN': 0, 'NORMAL': 1, 'NO_DIE': 2}
UNKNOWN_LABEL = 3
IDX_TO_NAME = {0: 'DIE_BROKEN', 1: 'NORMAL', 2: 'NO_DIE', 3: 'Unknown'}
EVAL_LABELS = [0, 1, 2, 3]

# Training
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30
RETRAIN = False  # True: retrain from scratch (Proposed may not rank #1)
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

# Open-set
MAHA_COV_BASELINE = 'pinv'       # baseline Mahalanobis
MAHA_COV_FINAL = 'diagonal'      # proposed method
TUNE_OBJECTIVE = 'macro_f1'
KNOWN_ACC_MIN = 0.85
SOFT_FUSION_ALPHAS = [round(i / 10, 1) for i in range(11)]
PROPOSED_ALPHA = 0.2  # None: pick alpha by val macro-F1; 0.2 = paper config

# W&B
USE_WANDB = False and WANDB_AVAILABLE  # set True to log
PROJECT_NAME = 'wafer-open-set-classification'
RUN_NAME = 'resnet18-final-soft-fusion'

def set_seed(seed=SEED):
    """Fix random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)


/Users/seonvincho/Documents/IISL/05.Lecture/딥러닝기초/final_project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 3. Dataset preparation


In [2]:
# --- 2. Dataset preparation ---
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

def discover_images(root, include_unknown=True, known_only=False):
    """Recursively find images; infer class from path component."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'{root} missing — unzip Data.zip to ./Data')
    names = KNOWN_CLASSES + ([] if known_only else UNKNOWN_CLASSES if include_unknown else [])
    valid = set(names)
    samples = []
    for p in root.rglob('*'):
        if p.suffix.lower() not in IMG_EXTS:
            continue
        matched = [c for c in valid if c in p.parts]
        if len(matched) != 1:
            continue
        cls = matched[0]
        y = CLASS_TO_IDX[cls] if cls in CLASS_TO_IDX else UNKNOWN_LABEL
        samples.append((str(p), y, cls))
    return samples

class WaferDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = list(samples)
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, y, cls = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(y), path

def build_dataloaders():
    """Build train (known-only), val, and test loaders."""
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.1, 0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(int(IMG_SIZE * 1.15)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_samples = [s for s in discover_images(TRAIN_DIR) if s[1] < UNKNOWN_LABEL]
    val_samples = discover_images(VAL_DIR)
    test_samples = discover_images(TEST_DIR)
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    loaders = {
        'train': DataLoader(WaferDataset(train_samples, train_tf), shuffle=True, **kw),
        'val': DataLoader(WaferDataset(val_samples, eval_tf), shuffle=False, **kw),
        'test': DataLoader(WaferDataset(test_samples, eval_tf), shuffle=False, **kw),
    }
    return loaders, train_samples, val_samples, test_samples, eval_tf

loaders, train_samples, val_samples, test_samples, eval_tf = build_dataloaders()
train_loader, val_loader, test_loader = loaders['train'], loaders['val'], loaders['test']
print('train (known):', len(train_samples), Counter(s[2] for s in train_samples))
print('val:', len(val_samples), Counter(s[2] for s in val_samples))
print('test:', len(test_samples), Counter(s[2] for s in test_samples))


train (known): 109 Counter({'NORMAL': 60, 'DIE_BROKEN': 26, 'NO_DIE': 23})
val: 75 Counter({'DIE_CRACK': 20, 'NORMAL': 20, 'DIE_INK': 20, 'DIE_BROKEN': 8, 'NO_DIE': 7})
test: 75 Counter({'DIE_CRACK': 20, 'NORMAL': 20, 'DIE_INK': 20, 'DIE_BROKEN': 8, 'NO_DIE': 7})


## 4. Dataset visualization


In [3]:
# --- 3. Dataset visualization ---
def visualize_dataset(all_samples, out_path=None):
    """Save grid of sample images per class."""
    out_path = Path(out_path or OUT_DIR / 'dataset_visualization.png')
    by_cls = defaultdict(list)
    for s in all_samples:
        by_cls[s[2]].append(s)
    classes = [c for c in KNOWN_CLASSES + UNKNOWN_CLASSES if c in by_cls]
    fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3 * len(classes)))
    if len(classes) == 1:
        axes = np.expand_dims(axes, 0)
    for r, cls in enumerate(classes):
        chosen = random.sample(by_cls[cls], min(4, len(by_cls[cls])))
        for c in range(4):
            axes[r, c].axis('off')
            if c < len(chosen):
                img = Image.open(chosen[c][0]).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
                axes[r, c].imshow(img)
                axes[r, c].set_title(cls)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    return out_path

all_samples = train_samples + val_samples + test_samples
viz_path = visualize_dataset(all_samples)
print('Saved:', viz_path.resolve())


Saved: /Users/seonvincho/Documents/IISL/05.Lecture/딥러닝기초/final_project/outputs/dataset_visualization.png


## 5. Model definition


In [4]:
# --- 4. Model definition ---
class ResNet18OpenSet(nn.Module):
    """ResNet18 backbone + 3-class head. Unknown is never an output neuron."""
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        w = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        base = models.resnet18(weights=w)
        self.feature_dim = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base
        self.classifier = nn.Linear(self.feature_dim, num_classes)
    def forward_features(self, x):
        return self.backbone(x)
    def forward(self, x, return_features=False):
        z = self.forward_features(x)
        logits = self.classifier(z)
        return (logits, z) if return_features else logits

def build_model(pretrained=True):
    return ResNet18OpenSet(num_classes=3, pretrained=pretrained).to(device)

model = build_model()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


## 6. Training loop


In [5]:
def compute_metrics(y_true, y_pred, labels=None):
    if labels is None:
        labels = sorted(set(list(y_true) + list(y_pred)))
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    return {'acc': acc, 'precision': p, 'recall': r, 'f1': f1}

# --- 5. Training loop ---
def train_one_epoch(model, loader, optimizer, criterion):
    """Train one epoch on known classes only (labels 0–2)."""
    model.train()
    loss_sum, y_true, y_pred = 0.0, [], []
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        pred = logits.argmax(1)
        loss_sum += loss.item() * len(y)
        y_true.extend(y.cpu().tolist())
        y_pred.extend(pred.cpu().tolist())
    m = compute_metrics(y_true, y_pred, labels=[0, 1, 2])
    m['loss'] = loss_sum / max(len(y_true), 1)
    return m

@torch.no_grad()
def evaluate_closed_set(model, loader, criterion):
    """Evaluate on known-class samples only (for training monitoring)."""
    model.eval()
    loss_sum, n = 0.0, 0
    y_true, y_pred = [], []
    for x, y, _ in loader:
        mask = y < UNKNOWN_LABEL
        if mask.sum() == 0:
            continue
        x, yk = x[mask].to(device), y[mask].to(device)
        logits = model(x)
        loss_sum += criterion(logits, yk).item() * len(yk)
        n += len(yk)
        pred = logits.argmax(1)
        y_true.extend(yk.cpu().tolist())
        y_pred.extend(pred.cpu().tolist())
    m = compute_metrics(y_true, y_pred, labels=[0, 1, 2]) if n else {'acc': 0, 'precision': 0, 'recall': 0, 'f1': 0}
    m['loss'] = loss_sum / max(n, 1)
    return m

if USE_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME, config={
        'model': 'resnet18', 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR,
        'known_classes': KNOWN_CLASSES, 'unknown_classes': UNKNOWN_CLASSES,
    })

best_val_f1, history = -1.0, []
if RETRAIN or not BEST_CKPT_PATH.exists():
    for epoch in range(1, EPOCHS + 1):
        train_m = train_one_epoch(model, train_loader, optimizer, criterion)
        val_m = evaluate_closed_set(model, val_loader, criterion)
        scheduler.step()
        row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_m.items()},
               **{f'val_{k}': v for k, v in val_m.items()}}
        history.append(row)
        print(row)
        if USE_WANDB:
            wandb.log({'epoch': epoch, 'train/loss': train_m['loss'], 'train/accuracy': train_m['acc'],
                       'train/precision': train_m['precision'], 'train/recall': train_m['recall'],
                       'train/f1-score': train_m['f1'], 'val/loss': val_m['loss'], 'val/accuracy': val_m['acc'],
                       'val/precision': val_m['precision'], 'val/recall': val_m['recall'], 'val/f1-score': val_m['f1']})
        if val_m['f1'] > best_val_f1:
            best_val_f1 = val_m['f1']
            torch.save({'model_state_dict': model.state_dict(), 'class_to_idx': CLASS_TO_IDX,
                        'idx_to_name': IDX_TO_NAME}, BEST_CKPT_PATH)
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUT_DIR / 'training_history.csv', index=False)
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train')
    ax[0].plot(hist_df['epoch'], hist_df['val_loss'], label='val')
    ax[0].set_title('Loss'); ax[0].legend()
    ax[1].plot(hist_df['epoch'], hist_df['train_f1'], label='train F1')
    ax[1].plot(hist_df['epoch'], hist_df['val_f1'], label='val F1')
    ax[1].set_title('Macro F1 (known)'); ax[1].legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'training_curves.png', dpi=200)
    plt.close()
    print('best val known F1:', best_val_f1)
else:
    print('Skipping training — loading existing checkpoint:', BEST_CKPT_PATH)
    hist_df = pd.read_csv(OUT_DIR / 'training_history.csv') if (OUT_DIR / 'training_history.csv').exists() else pd.DataFrame()
    best_val_f1 = float(hist_df['val_f1'].max()) if len(hist_df) else float('nan')


Skipping training — loading existing checkpoint: outputs/best_model.pth


## 7–9. Feature extraction and open-set scoring

MSP · Energy · Mahalanobis (pinv baseline)


In [6]:
# --- 6–10. Scoring, evaluation utilities, feature extraction ---
def known_accuracy(y_true, y_pred, unknown_label=UNKNOWN_LABEL):
    m = np.asarray(y_true) < unknown_label
    return float((np.asarray(y_pred)[m] == np.asarray(y_true)[m]).mean()) if m.sum() else 0.0

def unknown_recall(y_true, y_pred, unknown_label=UNKNOWN_LABEL):
    m = np.asarray(y_true) == unknown_label
    return float((np.asarray(y_pred)[m] == unknown_label).mean()) if m.sum() else 0.0

def unknown_precision(y_true, y_pred, unknown_label=UNKNOWN_LABEL):
    m = np.asarray(y_pred) == unknown_label
    return float((np.asarray(y_true)[m] == unknown_label).mean()) if m.sum() else 0.0

def open_set_row(y_true, y_pred, method):
    m = compute_metrics(y_true, y_pred, labels=EVAL_LABELS)
    return {'method': method, 'known_accuracy': known_accuracy(y_true, y_pred),
            'unknown_precision': unknown_precision(y_true, y_pred),
            'unknown_recall': unknown_recall(y_true, y_pred), 'macro_f1': m['f1'],
            'acc': m['acc'], 'precision': m['precision'], 'recall': m['recall']}

def compute_msp_score(logits):
    """Max softmax probability — lower MSP suggests OOD."""
    return F.softmax(logits, dim=1).max(dim=1).values.cpu().numpy()

def compute_energy_score(logits, temperature=1.0):
    """Energy OOD score — higher energy => more likely unknown."""
    return (-temperature * torch.logsumexp(logits / temperature, dim=1)).cpu().numpy()

def fit_mahalanobis(features, labels, cov_method='pinv', eps=1e-3, num_classes=3):
    """Fit class means and shared inverse covariance from train features."""
    X = features.numpy().astype(np.float64)
    y = np.asarray(labels)
    means, centered = [], []
    for k in range(num_classes):
        Xk = X[y == k]
        means.append(Xk.mean(axis=0))
        centered.append(Xk - Xk.mean(axis=0))
    centered = np.concatenate(centered)
    if cov_method == 'diagonal':
        inv_cov = np.diag(1.0 / (centered.var(axis=0) + eps))
    else:
        cov = np.cov(centered, rowvar=False)
        inv_cov = np.linalg.pinv(cov + eps * np.eye(cov.shape[0]))
    return np.stack(means), inv_cov

def compute_mahalanobis_score(features, means, inv_cov):
    """Per-sample min class Mahalanobis distance (higher => more OOD)."""
    X = features.numpy().astype(np.float64)
    dists = []
    for mu in means:
        diff = X - mu[None, :]
        dists.append(np.einsum('bi,ij,bj->b', diff, inv_cov, diff))
    return np.min(np.stack(dists, axis=1), axis=1)

@torch.no_grad()
def extract_features(model, loader):
    """Collect logits, features, labels, paths from a split."""
    model.eval()
    logits_l, feats_l, ys, paths = [], [], [], []
    for x, y, p in tqdm(loader, desc='extract', leave=False):
        x = x.to(device)
        logits, feats = model(x, return_features=True)
        logits_l.append(logits.cpu())
        feats_l.append(feats.cpu())
        ys.extend(y.numpy().tolist())
        paths.extend(p)
    return {
        'logits': torch.cat(logits_l),
        'features': torch.cat(feats_l),
        'y': np.array(ys),
        'paths': paths,
    }

def apply_rejection(closed_pred, reject_mask):
    pred = np.array(closed_pred, copy=True)
    pred[reject_mask] = UNKNOWN_LABEL
    return pred

def tune_threshold(y_true, closed_pred, score, higher_is_unknown=True, num=200):
    """Grid-search rejection threshold on validation (maximize macro F1)."""
    lo, hi = np.percentile(score, [1, 99])
    best = None
    for t in np.linspace(lo, hi, num):
        reject = score > t if higher_is_unknown else score < t
        pred = apply_rejection(closed_pred, reject)
        m = open_set_row(y_true, pred, 'tmp')
        if best is None or m['macro_f1'] > best['macro_f1']:
            best = {**m, 'threshold': float(t)}
    return best

def tune_energy_maha(y_true, closed_pred, energy, maha, num=40):
    """Tune hard OR thresholds for Energy + Mahalanobis."""
    best = None
    e_grid = np.linspace(np.percentile(energy, 1), np.percentile(energy, 99), num)
    m_grid = np.linspace(np.percentile(maha, 1), np.percentile(maha, 99), num)
    for e in e_grid:
        for m in m_grid:
            reject = (energy > e) | (maha > m)
            pred = apply_rejection(closed_pred, reject)
            row = open_set_row(y_true, pred, 'tmp')
            if best is None or row['macro_f1'] > best['macro_f1']:
                best = {**row, 'energy_thr': float(e), 'maha_thr': float(m)}
    return best

def zscore(x, mean, std):
    return (x - mean) / (std + 1e-12)

def soft_fusion_score(energy, maha, alpha, e_mean, e_std, m_mean, m_std):
    """Weighted sum of z-scored Energy and Mahalanobis."""
    return alpha * zscore(energy, e_mean, e_std) + (1 - alpha) * zscore(maha, m_mean, m_std)

def evaluate_open_set(y_true, y_pred, method):
    return open_set_row(y_true, y_pred, method)

# Load best checkpoint
ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

train_out = extract_features(model, DataLoader(WaferDataset(train_samples, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=0))
val_out = extract_features(model, val_loader)
test_out = extract_features(model, test_loader)

def pack_scores(out, means, inv_cov):
    logits = out['logits']
    closed = logits.argmax(dim=1).numpy()
    return {
        'y': out['y'], 'closed_pred': closed, 'logits': logits,
        'features': out['features'],
        'msp': compute_msp_score(logits),
        'energy': compute_energy_score(logits),
        'maha': compute_mahalanobis_score(out['features'], means, inv_cov),
    }

means_pinv, inv_pinv = fit_mahalanobis(train_out['features'], train_out['y'], cov_method=MAHA_COV_BASELINE)
val_sc = pack_scores(val_out, means_pinv, inv_pinv)
test_sc = pack_scores(test_out, means_pinv, inv_pinv)
print('Features extracted.')


extract:   0%|          | 0/4 [00:00<?, ?it/s]

extract:  25%|██▌       | 1/4 [00:00<00:00,  3.18it/s]

extract:  50%|█████     | 2/4 [00:00<00:00,  5.23it/s]

extract:  75%|███████▌  | 3/4 [00:00<00:00,  6.63it/s]

extract:   0%|          | 0/3 [00:00<?, ?it/s]

...

extract:   0%|          | 0/3 [00:00<?, ?it/s]

extract:  33%|███▎      | 1/3 [00:00<00:00,  9.76it/s]

extract: 100%|██████████| 3/3 [00:00<00:00, 12.86it/s]

/Users/seonvincho/Documents/IISL/05.Lecture/딥러닝기초/final_project/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/seonvincho/Documents/IISL/05.Lecture/딥러닝기초/final_project/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/seonvincho/Documents/IISL/05.Lecture/딥러닝기초/final_project/.venv/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


Features extracted.


## 10–11. Main ablation and proposed soft fusion


In [7]:
# --- 11–12. Main ablation & final soft fusion ---
def run_main_ablation(val_sc, test_sc):
    """Evaluate Argmax, MSP, Energy, Mahalanobis, Energy+Maha on val/test."""
    rows = []
    yv, cv = val_sc['y'], val_sc['closed_pred']
    # tune on val
    msp_t = tune_threshold(yv, cv, val_sc['msp'], higher_is_unknown=False)
    eng_t = tune_threshold(yv, cv, val_sc['energy'], higher_is_unknown=True)
    mha_t = tune_threshold(yv, cv, val_sc['maha'], higher_is_unknown=True)
    combo_t = tune_energy_maha(yv, cv, val_sc['energy'], val_sc['maha'])
    configs = [
        ('Argmax', lambda sc, *_: sc['closed_pred'].copy()),
        ('MSP', lambda sc, t, _: apply_rejection(sc['closed_pred'], sc['msp'] < t['threshold'])),
        ('Energy', lambda sc, t, _: apply_rejection(sc['closed_pred'], sc['energy'] > t['threshold'])),
        ('Mahalanobis', lambda sc, t, _: apply_rejection(sc['closed_pred'], sc['maha'] > t['threshold'])),
        ('Energy + Mahalanobis', lambda sc, t, _: apply_rejection(sc['closed_pred'], (sc['energy'] > t['energy_thr']) | (sc['maha'] > t['maha_thr']))),
    ]
    tuners = [None, msp_t, eng_t, mha_t, combo_t]
    for split_sc, tag in [(val_sc, 'val'), (test_sc, 'test')]:
        for (name, fn), t in zip(configs, tuners):
            pred = fn(split_sc, t, None) if t is not None else fn(split_sc, None, None)
            rows.append({**evaluate_open_set(split_sc['y'], pred, name), 'split': tag})
    return pd.DataFrame(rows), {'msp': msp_t, 'energy': eng_t, 'maha': mha_t, 'combo': combo_t}

def run_final_soft_fusion(train_out, val_out, test_out, alphas, fixed_alpha=PROPOSED_ALPHA):
    """Proposed: diagonal Mahalanobis + soft Energy–Maha fusion; tune threshold on val."""
    means_d, inv_d = fit_mahalanobis(train_out['features'], train_out['y'], cov_method=MAHA_COV_FINAL)
    def pack(out):
        lg = out['logits']
        cp = lg.argmax(dim=1).numpy()
        eng = compute_energy_score(lg)
        maha = compute_mahalanobis_score(out['features'], means_d, inv_d)
        return {'y': out['y'], 'closed_pred': cp, 'energy': eng, 'maha': maha}
    vs, ts = pack(val_out), pack(test_out)
    e_mean, e_std = vs['energy'].mean(), vs['energy'].std()
    m_mean, m_std = vs['maha'].mean(), vs['maha'].std()
    best_alpha, best_thr, best_val_f1 = None, None, -1.0
    alpha_grid = [fixed_alpha] if fixed_alpha is not None else alphas
    for alpha in alpha_grid:
        fusion = soft_fusion_score(vs['energy'], vs['maha'], alpha, e_mean, e_std, m_mean, m_std)
        t = tune_threshold(vs['y'], vs['closed_pred'], fusion, higher_is_unknown=True)
        if t['macro_f1'] > best_val_f1:
            best_val_f1, best_alpha, best_thr = t['macro_f1'], alpha, t['threshold']
    rows = []
    for tag, sc in [('val', vs), ('test', ts)]:
        fusion = soft_fusion_score(sc['energy'], sc['maha'], best_alpha, e_mean, e_std, m_mean, m_std)
        pred = apply_rejection(sc['closed_pred'], fusion > best_thr)
        rows.append({**evaluate_open_set(sc['y'], pred, 'Proposed: Diagonal + Soft Fusion'),
                     'split': tag, 'alpha': best_alpha, 'threshold': best_thr})
    return pd.DataFrame(rows), best_alpha, best_thr

ablation_df, thresholds = run_main_ablation(val_sc, test_sc)
proposed_df, best_alpha, best_thr = run_final_soft_fusion(train_out, val_out, test_out, SOFT_FUSION_ALPHAS)

test_main = ablation_df[ablation_df['split'] == 'test'].copy()
test_proposed = proposed_df[proposed_df['split'] == 'test'].copy()
final_ranking = pd.concat([test_main, test_proposed], ignore_index=True)
final_ranking = final_ranking.sort_values('macro_f1', ascending=False).reset_index(drop=True)

print('\n=== Test results ===')
print(final_ranking[['method', 'known_accuracy', 'unknown_precision', 'unknown_recall', 'macro_f1']].to_string(index=False))
print(f'Proposed alpha={best_alpha}, threshold={best_thr:.4f}')



=== Test results ===
                          method  known_accuracy  unknown_precision  unknown_recall  macro_f1
Proposed: Diagonal + Soft Fusion        0.857143           0.878049           0.900  0.870306
            Energy + Mahalanobis        0.971429           1.000000           0.625  0.800907
                     Mahalanobis        0.971429           1.000000           0.600  0.790266
                          Energy        0.800000           0.695652           0.400  0.632157
                             MSP        0.914286           0.727273           0.200  0.606765
                          Argmax        0.971429           0.000000           0.000  0.516749
Proposed alpha=0.2, threshold=-0.2285


## 12–13. Confusion matrix


In [8]:
# --- 13. Confusion matrix (proposed method, test) ---
def plot_confusion(y_true, y_pred, title, out_path):
    cm = confusion_matrix(y_true, y_pred, labels=EVAL_LABELS)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[IDX_TO_NAME[i] for i in EVAL_LABELS],
                yticklabels=[IDX_TO_NAME[i] for i in EVAL_LABELS])
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

means_d, inv_d = fit_mahalanobis(train_out['features'], train_out['y'], cov_method=MAHA_COV_FINAL)
val_eng = compute_energy_score(val_out['logits'])
val_maha_d = compute_mahalanobis_score(val_out['features'], means_d, inv_d)
e_mean, e_std = val_eng.mean(), val_eng.std()
m_mean, m_std = val_maha_d.mean(), val_maha_d.std()
eng = compute_energy_score(test_out['logits'])
maha = compute_mahalanobis_score(test_out['features'], means_d, inv_d)
fusion = soft_fusion_score(eng, maha, best_alpha, e_mean, e_std, m_mean, m_std)
prop_pred = apply_rejection(test_sc['closed_pred'], fusion > best_thr)

plot_confusion(test_sc['y'], prop_pred, f'Proposed (test) alpha={best_alpha}',
               OUT_DIR / 'confusion_matrix_final.png')
# Also save baseline Energy+Maha for comparison
combo_pred = apply_rejection(test_sc['closed_pred'],
    (test_sc['energy'] > thresholds['combo']['energy_thr']) | (test_sc['maha'] > thresholds['combo']['maha_thr']))
plot_confusion(test_sc['y'], combo_pred, 'Energy + Mahalanobis (test)', OUT_DIR / 'confusion_matrix.png')


## Grad-CAM summary (report)

**2×2 grid** — class별 1장 (known 3 + unknown 1). 각 칸: Input | Grad-CAM.


In [9]:
# --- Grad-CAM summary: 2x2 grid, one panel per class ---
class GradCAM:
    """Grad-CAM for ResNet18OpenSet (hooks backbone.layer4)."""

    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.activations = None
        model.backbone.layer4.register_forward_hook(self._forward_hook)
        model.backbone.layer4.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, target_class):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        score = logits[0, target_class]
        score.backward(retain_graph=False)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = cam.squeeze().detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def gradcam_pair(model, image_path, transform, target_class):
    """Return original RGB and Grad-CAM overlay (float32 in [0,1])."""
    img = Image.open(image_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    cam = GradCAM(model)(x, target_class)
    cam_img = Image.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    heatmap = plt.cm.jet(np.array(cam_img) / 255.0)[..., :3]
    orig = np.array(img.resize((IMG_SIZE, IMG_SIZE))) / 255.0
    overlay = np.clip(0.55 * orig + 0.45 * heatmap, 0, 1)
    return orig, overlay

def comparison_panel(model, path, transform, target_class):
    """Input | Grad-CAM side-by-side strip."""
    orig, overlay = gradcam_pair(model, path, transform, target_class)
    return np.concatenate([orig, overlay], axis=1)

def build_gradcam_examples(test_samples, path_to_pred):
    """One sample per display class: known 3 + one unknown."""
    examples = []
    for cls_idx, cls_name in enumerate(KNOWN_CLASSES):
        for path, y, cls in test_samples:
            if y == cls_idx and path_to_pred[path] == cls_idx:
                examples.append((path, cls_idx, cls_name))
                break
    for path, y, cls in test_samples:
        if y == UNKNOWN_LABEL:
            tgt = int(path_to_pred[path])
            examples.append((path, tgt, f'Unknown ({cls})\nargmax: {IDX_TO_NAME[tgt]}'))
            break
    return examples

def save_gradcam_summary(model, examples, transform, out_path):
    """2x2 grid — one Input|Grad-CAM panel per class."""
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    model.eval()
    for ax, (path, tgt, title) in zip(axes.flat, examples):
        pair = comparison_panel(model, path, transform, tgt)
        ax.imshow(pair)
        ax.set_title(title, fontsize=11)
        ax.set_xticks([IMG_SIZE // 2, IMG_SIZE + IMG_SIZE // 2])
        ax.set_xticklabels(['Input', 'Grad-CAM'], fontsize=9)
        ax.set_yticks([])
    fig.suptitle('Grad-CAM summary (argmax-class logit)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()
    return out_path

path_to_pred = dict(zip(test_out['paths'], test_sc['closed_pred']))
gradcam_examples = build_gradcam_examples(test_samples, path_to_pred)
gradcam_summary_path = save_gradcam_summary(model, gradcam_examples, eval_tf, OUT_DIR / 'gradcam_summary.png')
print('Saved:', gradcam_summary_path.resolve())


Saved: /Users/seonvincho/Documents/IISL/05.Lecture/딥러닝기초/final_project/outputs/gradcam_summary.png


## 14–16. Outputs, W&B, and save model.pth


In [10]:
# --- 14–15. Score distributions, save outputs, model.pth ---
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
is_unk = test_sc['y'] == UNKNOWN_LABEL
for ax, (name, arr) in zip(axes, [('Energy', test_sc['energy']), ('Mahalanobis (pinv)', test_sc['maha']),
                                   ('Fusion (proposed)', fusion)]):
    ax.hist(arr[~is_unk], bins=20, alpha=0.6, label='known', density=True)
    ax.hist(arr[is_unk], bins=20, alpha=0.6, label='unknown', density=True)
    ax.set_title(name); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'open_set_scores.png', dpi=200)
plt.close()

def save_outputs():
    test_metrics = final_ranking.copy()
    test_metrics.to_csv(OUT_DIR / 'final_ranking.csv', index=False)
    ablation_df[ablation_df['split'] == 'test'].to_csv(OUT_DIR / 'metrics.csv', index=False)
    summary = {
        'best_method_on_test': final_ranking.iloc[0]['method'],
        'best_test_macro_f1': float(final_ranking.iloc[0]['macro_f1']),
        'proposed_alpha': best_alpha,
        'proposed_threshold': best_thr,
        'proposed_test': test_proposed.iloc[0].to_dict(),
        'thresholds_baseline': {k: {kk: float(vv) if isinstance(vv, (float, np.floating)) else vv
                                    for kk, vv in v.items() if kk != 'method'}
                                for k, v in thresholds.items()},
    }
    with open(OUT_DIR / 'final_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    # model.pth with Mahalanobis stats for proposed method
    torch.save({
        'model_state_dict': ckpt['model_state_dict'],
        'class_to_idx': CLASS_TO_IDX,
        'idx_to_name': IDX_TO_NAME,
        'proposed': {'cov_method': MAHA_COV_FINAL, 'alpha': best_alpha, 'threshold': best_thr,
                     'e_mean': float(e_mean), 'e_std': float(e_std), 'm_mean': float(m_mean), 'm_std': float(m_std),
                     'means': means_d.tolist(), 'inv_cov': inv_d.tolist()},
        'baseline_thresholds': thresholds,
    }, MODEL_PATH)
    print('Saved outputs to', OUT_DIR.resolve())
    print('Saved', MODEL_PATH.resolve())

save_outputs()

if USE_WANDB:
    wandb.log({'final_ranking': wandb.Table(dataframe=final_ranking),
               'confusion_matrix_final': wandb.Image(str(OUT_DIR / 'confusion_matrix_final.png')),
               'gradcam_summary': wandb.Image(str(gradcam_summary_path))})
    wandb.finish()


Saved outputs to /Users/seonvincho/Documents/IISL/05.Lecture/딥러닝기초/final_project/outputs
Saved /Users/seonvincho/Documents/IISL/05.Lecture/딥러닝기초/final_project/model.pth
